# Amazon ML Challenge 2026 — Business Entity Resolution

**Pipeline:** Normalize → Block (candidate generation) → ML Match (LightGBM) → Evaluate → Submit

---
## STEP 0 — Setup
Mount Google Drive and install dependencies.

**Upload your dataset to Google Drive first** with this structure:
```
MyDrive/
  amazon_ml/
    dataset/
      train/
        train_source1.tsv, train_source2.tsv, train_source3.tsv, train_ground_truth.tsv
      test/
        test_source1.tsv, test_source2.tsv, test_source3.tsv
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# CONFIGURE THIS — path to the folder containing dataset/ in your Google Drive
BASE_DIR = '/content/drive/MyDrive/amazon_ml'

DATA_DIR   = os.path.join(BASE_DIR, 'dataset')
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
NORM_DIR   = os.path.join(BASE_DIR, 'artifacts', 'normalized')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

os.makedirs(NORM_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Drive mounted and directories ready')
print(f'  Data    : {DATA_DIR}')
print(f'  Norm    : {NORM_DIR}')
print(f'  Output  : {OUTPUT_DIR}')

In [ ]:
!pip install -q pyarrow pandas lightgbm rapidfuzz scikit-learn

---
## Core Source Code (inlined from src/)

In [ ]:
# io_utils
import csv, os
import pandas as pd

DELIM = '\t'

def read_tsv(path):
    return pd.read_csv(path, sep=DELIM, dtype=str,
                       keep_default_na=False, na_filter=False,
                       quoting=csv.QUOTE_NONE, encoding='utf-8')

def parse_ids(cell):
    return [x for x in cell.split(',') if x] if cell else []

def load_ground_truth(path):
    df = read_tsv(path)
    return {row.source1_entity_id: parse_ids(row.matched_entity_ids) for row in df.itertuples()}

def write_output(path, s1_ids, mapping, col):
    rows = [(s1, ','.join(list(dict.fromkeys(mapping.get(s1, []))))) for s1 in s1_ids]
    out_df = pd.DataFrame(rows, columns=['source1_entity_id', col])
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    out_df.to_csv(path, sep=DELIM, index=False, quoting=csv.QUOTE_NONE)

print('io_utils OK')

In [ ]:
# normalize
import re, unicodedata

_WS_RE = re.compile(r'\s+')
_PUNCT_KEEP_RE = re.compile(r'[^a-z0-9/\- ]')

def strip_accents(text):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', text) if not unicodedata.combining(ch))

def clean_text(text):
    text = strip_accents(text).lower().replace('&', ' and ')
    return _WS_RE.sub(' ', _PUNCT_KEEP_RE.sub(' ', text)).strip()

LEGAL_FORMS = ['incorporated','inc','llc','ltd','limited','corporation','corp',
    'co','company','pvt','private','plc','llp','lp','pllc',
    'gmbh','sa','sas','sasu','sarl','eurl','sci','snc','ste','societe','ets','etablissements']
_LEGAL_FORM_RE = re.compile(r'\b(' + '|'.join(sorted(LEGAL_FORMS, key=len, reverse=True)) + r')\b')

def split_legal_form(name_clean):
    matches = _LEGAL_FORM_RE.findall(name_clean)
    if not matches: return name_clean, ''
    return _WS_RE.sub(' ', _LEGAL_FORM_RE.sub(' ', name_clean)).strip(), ' '.join(matches)

def tokens_sorted(text): return ' '.join(sorted(text.split()))

def acronym(name_core):
    t = name_core.split()
    return ''.join(x[0] for x in t if x) if len(t) >= 2 else ''

def normalize_name(raw):
    nc = clean_text(raw)
    core, lf = split_legal_form(nc)
    return {'name_clean': nc, 'name_core': core, 'legal_form': lf,
            'name_tokens_sorted': tokens_sorted(core), 'acronym': acronym(core)}

ADDR_ABBREV = {'rd':'road','st':'street','ave':'avenue','av':'avenue','blvd':'boulevard',
    'bd':'boulevard','ln':'lane','dr':'drive','apt':'apartment','ste':'suite',
    'fl':'floor','opp':'opposite','nr':'near','bldg':'building','mkt':'market',
    'r':'rue','pl':'place','fbg':'faubourg','chem':'chemin','imp':'impasse','all':'allee'}
_ABBREV_RE   = re.compile(r'\b(' + '|'.join(sorted(ADDR_ABBREV, key=len, reverse=True)) + r')\b')
_LANDMARK_RE = re.compile(r'\b(near|opp|opposite|behind|beside|next to|pres de|en face)\b(.*)$')
_POSTAL_RE   = re.compile(r'\b(\d{5,6})\b')
_NUMBER_RE   = re.compile(r'\d+')

def normalize_address(raw):
    ac = _ABBREV_RE.sub(lambda m: ADDR_ABBREV[m.group(1)], clean_text(raw))
    m  = _LANDMARK_RE.search(ac)
    lm = m.group(0).strip() if m else ''
    anl = ac[:m.start()].strip() if m else ac
    postal_m = _POSTAL_RE.findall(ac)
    return {'addr_clean': ac, 'addr_no_landmark': anl, 'landmark': lm,
            'postal': postal_m[-1] if postal_m else '',
            'numbers': sorted(set(_NUMBER_RE.findall(ac)))}

print('normalize OK')

In [ ]:
# evaluate
def f05_entity(pred, true, beta=0.5):
    pred, true = set(pred), set(true)
    if not true and not pred: return 1.0
    if not true or not pred:  return 0.0
    tp = len(pred & true)
    if tp == 0: return 0.0
    p, r, b2 = tp/len(pred), tp/len(true), beta*beta
    return (1+b2)*p*r / (b2*p + r)

def macro_f05(pred_map, true_map):
    if not true_map: return 0.0
    return sum(f05_entity(pred_map.get(s1,[]),m) for s1,m in true_map.items()) / len(true_map)

assert abs(f05_entity(['S2-00047','S2-00193','S3-00812'],['S2-00047','S3-00812']) - 0.714) < 1e-3
print('evaluate OK — self-check passed (0.714)')

---
## STEP 1 — Normalize (Build Parquet Cache)
Chunks large files to stay within Colab RAM. Skips already-done files.

In [ ]:
import time

FILES = [('train','source1'),('train','source2'),('train','source3'),
         ('test', 'source1'),('test', 'source2'),('test', 'source3')]
CHUNK_SIZE = 500_000

def normalize_file(split, source):
    out_path = os.path.join(NORM_DIR, f'{split}_{source}.parquet')
    if os.path.exists(out_path):
        print(f'  SKIP {split}/{source} — already exists'); return
    path = os.path.join(DATA_DIR, split, f'{split}_{source}.tsv')
    df   = read_tsv(path)
    total = len(df)
    print(f'  {split}/{source}: {total:,} rows...')
    chunks, t0 = [], time.time()
    for start in range(0, total, CHUNK_SIZE):
        c = df.iloc[start:start+CHUNK_SIZE]
        nv = pd.DataFrame(c['business_name'].map(normalize_name).tolist(), index=c.index)
        av = pd.DataFrame(c['business_address'].map(normalize_address).tolist(), index=c.index)
        av['numbers'] = av['numbers'].map(lambda xs: ','.join(xs))
        chunks.append(pd.concat([c[['entity_id','country']], nv, av], axis=1))
        print(f'    {min(start+CHUNK_SIZE,total)/total*100:.0f}%...', end='\r')
    out = pd.concat(chunks, ignore_index=True)
    out.to_parquet(out_path, index=False)
    print(f'  DONE {split}/{source}: {total:,} rows → {time.time()-t0:.1f}s')

t0 = time.time()
for split, source in FILES:
    normalize_file(split, source)
print(f'\nTotal: {time.time()-t0:.1f}s')

---
## STEP 2 — Blocking (Candidate Generation)
Rare-token inverted index + postal code exact match, per country.
**Improved params**: DF cutoff 200 (was 50), cap 100 (was 30).

In [ ]:
from collections import defaultdict
import time

RARE_TOKEN_DF_CUTOFF = 200  # tokens appearing in <= this many records are indexed
K_FINAL = 100               # max candidates per S1 entity
BLOCKING_COLS = ['entity_id','country','name_core','postal']

def load_norm(split, source):
    return pd.read_parquet(os.path.join(NORM_DIR, f'{split}_{source}.parquet'), columns=BLOCKING_COLS)

def build_token_index(pool):
    df_counts, row_tokens = defaultdict(int), []
    for nc in pool['name_core']:
        toks = set(str(nc).split())
        row_tokens.append(toks)
        for t in toks: df_counts[t] += 1
    index = defaultdict(list)
    for i, toks in enumerate(row_tokens):
        for t in toks:
            if df_counts[t] <= RARE_TOKEN_DF_CUTOFF: index[t].append(i)
    return index

def build_postal_index(pool):
    idx = defaultdict(list)
    for i, p in enumerate(pool['postal']):
        if p: idx[p].append(i)
    return idx

def block_country(s1_g, pool_g):
    tok_idx, post_idx = build_token_index(pool_g), build_postal_index(pool_g)
    pool_ids = pool_g['entity_id'].tolist()
    out = {}
    for row in s1_g.itertuples():
        hits = set()
        for t in set(str(row.name_core).split()): hits.update(tok_idx.get(t,()))
        if row.postal: hits.update(post_idx.get(row.postal,()))
        cands = [pool_ids[i] for i in hits]
        out[row.entity_id] = cands[:K_FINAL] if len(cands)>K_FINAL else cands
    return out

def run_blocking(split):
    print(f'Blocking {split}...')
    s1 = load_norm(split,'source1')
    s2 = load_norm(split,'source2'); s3 = load_norm(split,'source3')
    pool = pd.concat([s2,s3],ignore_index=True); del s2,s3
    all_cands = {}
    for country, s1g in s1.groupby('country'):
        pg = pool[pool['country']==country].reset_index(drop=True)
        t0 = time.time()
        c  = block_country(s1g, pg)
        all_cands.update(c)
        n  = sum(len(v) for v in c.values())
        print(f'  {split}/{country}: {len(s1g):,} S1 | pool {len(pg):,} | {n:,} pairs | {time.time()-t0:.1f}s')
    return all_cands

print('Blocking functions defined')

In [ ]:
# Run blocking on train + measure recall
train_cands = run_blocking('train')

gt_path  = os.path.join(TRAIN_DIR, 'train_ground_truth.tsv')
true_map = load_ground_truth(gt_path)
gt_df    = read_tsv(gt_path)

total_p = found_p = full_ent = non_sing = 0
for row in gt_df.itertuples():
    true = set(parse_ids(row.matched_entity_ids))
    if not true: continue
    non_sing += 1
    cand = set(train_cands.get(row.source1_entity_id, []))
    hit  = true & cand
    total_p += len(true); found_p += len(hit)
    if hit == true: full_ent += 1

print(f'\nBLOCKING RECALL')
print(f'  Pair recall        : {found_p:,}/{total_p:,} = {found_p/total_p:.4%}')
print(f'  Entity full-recall : {full_ent:,}/{non_sing:,} = {full_ent/non_sing:.4%}')

---
## STEP 3 — ML Matching Model (LightGBM)
Train on string similarity features to distinguish true matches from false candidates.

In [ ]:
from rapidfuzz import fuzz
import numpy as np

FEAT_NAMES = ['name_token_set','name_token_sort','name_ratio','name_sorted_ts',
              'acronym_match','legal_form_match',
              'addr_token_set','addr_no_lm_sort','postal_exact','num_jaccard']

def feats(r1, r2):
    n1,n2 = str(r1['name_core']),str(r2['name_core'])
    a1,a2 = str(r1['addr_clean']),str(r2['addr_clean'])
    al1,al2 = str(r1['addr_no_landmark']),str(r2['addr_no_landmark'])
    num1 = set(str(r1['numbers']).split(','))
    num2 = set(str(r2['numbers']).split(','))
    return [
        fuzz.token_set_ratio(n1,n2)/100,
        fuzz.token_sort_ratio(n1,n2)/100,
        fuzz.ratio(n1,n2)/100,
        fuzz.token_set_ratio(str(r1['name_tokens_sorted']),str(r2['name_tokens_sorted']))/100,
        int(r1['acronym']!='' and r1['acronym']==r2['acronym']),
        int(r1['legal_form']==r2['legal_form']),
        fuzz.token_set_ratio(a1,a2)/100,
        fuzz.token_sort_ratio(al1,al2)/100,
        int(r1['postal']!='' and r1['postal']==r2['postal']),
        len(num1&num2)/max(1,len(num1|num2)),
    ]

print(f'Features defined: {FEAT_NAMES}')

In [ ]:
# Load full normalized data for feature extraction
print('Loading full train normalized data...')
t0 = time.time()
s1n  = pd.read_parquet(os.path.join(NORM_DIR,'train_source1.parquet'))
s2n  = pd.read_parquet(os.path.join(NORM_DIR,'train_source2.parquet'))
s3n  = pd.read_parquet(os.path.join(NORM_DIR,'train_source3.parquet'))
pooln = pd.concat([s2n,s3n],ignore_index=True); del s2n,s3n
print(f'Loaded in {time.time()-t0:.1f}s')

s1_lkp   = s1n.set_index('entity_id').to_dict('index')
pool_lkp = pooln.set_index('entity_id').to_dict('index')
del pooln
print(f'Lookups: S1={len(s1_lkp):,}  pool={len(pool_lkp):,}')

In [ ]:
# Build training feature matrix
SAMPLE_NEG = 5   # negatives per positive
rng = np.random.default_rng(42)
X_rows, y_rows = [], []
t0 = time.time()

for s1_id, cand_ids in train_cands.items():
    true_ids = set(true_map.get(s1_id, []))
    if not cand_ids: continue
    r1 = s1_lkp.get(s1_id)
    if r1 is None: continue
    pos = [c for c in cand_ids if c in true_ids]
    neg = [c for c in cand_ids if c not in true_ids]
    for c in pos:
        r2 = pool_lkp.get(c)
        if r2: X_rows.append(feats(r1,r2)); y_rows.append(1)
    n_neg = min(len(neg), len(pos)*SAMPLE_NEG)
    if n_neg > 0:
        for c in rng.choice(neg, size=n_neg, replace=False):
            r2 = pool_lkp.get(c)
            if r2: X_rows.append(feats(r1,r2)); y_rows.append(0)

X = np.array(X_rows, dtype=np.float32)
y = np.array(y_rows, dtype=np.int32)
print(f'Feature matrix: {X.shape}  positives={y.sum():,}  negatives={(y==0).sum():,}  time={time.time()-t0:.1f}s')

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

Xtr,Xv,ytr,yv = train_test_split(X,y,test_size=0.1,random_state=42,stratify=y)

model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=63,
    min_child_samples=20, scale_pos_weight=(y==0).sum()/max(1,y.sum()),
    random_state=42, n_jobs=-1
)
model.fit(Xtr,ytr, eval_set=[(Xv,yv)],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(100)])

print('\nValidation Report:')
print(classification_report(yv, model.predict(Xv), target_names=['no-match','match']))

In [ ]:
# Feature importance plot
import matplotlib.pyplot as plt
import numpy as np

imp = model.feature_importances_
idx = np.argsort(imp)[::-1]
plt.figure(figsize=(9,4))
plt.bar(range(len(FEAT_NAMES)), imp[idx], color='steelblue')
plt.xticks(range(len(FEAT_NAMES)), [FEAT_NAMES[i] for i in idx], rotation=40, ha='right')
plt.title('Feature Importance'); plt.tight_layout(); plt.show()

---
## STEP 4 — Local F0.5 Evaluation

In [ ]:
THRESHOLD = 0.5  # tune this — try 0.3 to 0.8

def predict_matches(s1_ids, cands_dict, s1_lkp, pool_lkp, clf, thr):
    pred = {}
    for s1 in s1_ids:
        cands = cands_dict.get(s1, [])
        r1 = s1_lkp.get(s1)
        if not r1 or not cands: pred[s1] = []; continue
        valid = [(c, pool_lkp.get(c)) for c in cands if pool_lkp.get(c) is not None]
        if not valid: pred[s1] = []; continue
        Xb = np.array([feats(r1,r2) for _,r2 in valid], dtype=np.float32)
        probs = clf.predict_proba(Xb)[:,1]
        pred[s1] = [c for (c,_),p in zip(valid,probs) if p>=thr]
    return pred

print('Running train inference...')
t0 = time.time()
train_pred = predict_matches(list(true_map.keys()), train_cands, s1_lkp, pool_lkp, model, THRESHOLD)
print(f'Done in {time.time()-t0:.1f}s')

f05 = macro_f05(train_pred, true_map)
n   = len(true_map)
sing_frac = sum(1 for v in true_map.values() if not v) / n
print(f'\nLOCAL F0.5 SCORE (threshold={THRESHOLD})')
print(f'  Macro F0.5         : {f05:.6f}')
print(f'  N entities         : {n:,}')
print(f'  Singleton baseline : {sing_frac:.4%}  (trivial always-empty score)')

In [ ]:
# Quick threshold sweep on a 10k sample
sample_ids = list(true_map.keys())[:10000]
sample_true = {k: true_map[k] for k in sample_ids}

print('Threshold sweep (10k sample):')
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    p = predict_matches(sample_ids, train_cands, s1_lkp, pool_lkp, model, thr)
    score = macro_f05(p, sample_true)
    print(f'  threshold={thr:.1f} → F0.5={score:.4f}')

---
## STEP 5 — Generate Test Submission

In [ ]:
# Block test
test_cands = run_blocking('test')
print(f'Total test candidate pairs: {sum(len(v) for v in test_cands.values()):,}')

In [ ]:
# Load test normalized data
print('Loading test normalized data...')
t0 = time.time()
ts1  = pd.read_parquet(os.path.join(NORM_DIR,'test_source1.parquet'))
ts2  = pd.read_parquet(os.path.join(NORM_DIR,'test_source2.parquet'))
ts3  = pd.read_parquet(os.path.join(NORM_DIR,'test_source3.parquet'))
tpool = pd.concat([ts2,ts3],ignore_index=True); del ts2,ts3
print(f'Loaded in {time.time()-t0:.1f}s')

ts1_lkp   = ts1.set_index('entity_id').to_dict('index')
tpool_lkp = tpool.set_index('entity_id').to_dict('index')
del tpool
test_s1_ids = ts1['entity_id'].tolist()
print(f'Test S1: {len(test_s1_ids):,} entities')

In [ ]:
# Run inference
print('Running test inference...')
t0 = time.time()
test_pred = predict_matches(test_s1_ids, test_cands, ts1_lkp, tpool_lkp, model, THRESHOLD)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Predicted {sum(1 for v in test_pred.values() if v):,}/{len(test_s1_ids):,} entities have matches')

In [ ]:
# Write submission files
matching_path   = os.path.join(OUTPUT_DIR,'matching_results.tsv')
candidates_path = os.path.join(OUTPUT_DIR,'candidate_pairs.tsv')

write_output(matching_path,   test_s1_ids, test_pred,   'matched_entity_ids')
write_output(candidates_path, test_s1_ids, test_cands,  'candidate_entity_ids')

print(f'Written:')
print(f'  {matching_path}')
print(f'  {candidates_path}')

---
## STEP 6 — Validate & Download

In [ ]:
# Inline validation
def validate(m_path, c_path, test_dir):
    issues = []
    m_df = read_tsv(m_path)
    s1_df   = read_tsv(os.path.join(test_dir,'test_source1.tsv'))
    s2_df   = read_tsv(os.path.join(test_dir,'test_source2.tsv'))
    s3_df   = read_tsv(os.path.join(test_dir,'test_source3.tsv'))
    valid_s1   = set(s1_df['entity_id'])
    valid_pool = set(s2_df['entity_id']) | set(s3_df['entity_id'])
    submitted  = set(m_df['source1_entity_id'])
    missing    = valid_s1 - submitted
    if missing: issues.append(f'Missing {len(missing):,} S1 entities')
    if m_df['source1_entity_id'].duplicated().any(): issues.append('Duplicate S1 rows')
    for _, row in m_df.iterrows():
        bad = [i for i in parse_ids(row.get('matched_entity_ids','')) if i not in valid_pool]
        if bad: issues.append(f'Invalid pool IDs: {bad[:3]}'); break
    if issues:
        print('FAILED:'); [print(f'  {i}. {x}') for i,x in enumerate(issues,1)]
    else:
        print(f'PASS — {len(m_df):,} rows in matching_results.tsv')

validate(matching_path, candidates_path, TEST_DIR)

In [ ]:
# Download submission file
from google.colab import files
files.download(matching_path)
print('matching_results.tsv downloaded — upload this to the leaderboard portal!')